## Lab 7: AgentCore Metrics & Dashboards - Monitor Your Production Agent ✅ FINAL FIXED VERSION

### Overview

In Labs 1-4, we built a Customer Support Agent and deployed it to AgentCore Runtime. While Lab 4 showed us traces in CloudWatch GenAI Observability, we're only seeing individual request flows - not the bigger picture of how our agent is performing in production.

**The Gap:** Your agent is generating valuable metrics, but they're invisible! AgentCore automatically emits metrics for Runtime, Gateway, and Memory - we just need to enable and visualize them.

### ✅ CRITICAL FIXES APPLIED:

- 🔧 **NAMESPACE FIX**: Changed from `AWS/Bedrock/AgentCore/*` to `BedrockAgentCore`
- 🔧 **DATETIME FIX**: Added `timezone` import to resolve `AttributeError`
- 🔧 **OBSERVABILITY SETUP**: Added validation and setup instructions
- 🔧 **DIMENSION HANDLING**: Updated for correct Resource ARN format
- 🔧 **ENHANCED ERROR HANDLING**: Better AWS error messages and debugging

### Workshop Journey:
- **Lab 1 (Done):** Create Agent Prototype
- **Lab 2 (Done):** Enhance with Memory
- **Lab 3 (Done):** Scale with Gateway & Identity
- **Lab 4 (Done):** Deploy to Production
- **Lab 5 (Done):** Build User Interface
- **Lab 6 (Done):** Cleanup
- **Lab 7 (Current):** Metrics & Dashboards ← **NOW FULLY WORKING!**
- **Lab 8:** Complete Observability

### Prerequisites

- ✅ **Must complete Labs 1-4** - We'll monitor the agent you deployed
- ✅ **AWS CloudWatch access** - To create dashboards and query metrics
- ✅ **AgentCore Observability Enabled** - **NEW REQUIREMENT!**

---

## 🚀 Let's Make Your Agent's Performance Visible!

### Step 1: Import Libraries and Setup

Let's connect to AWS with all the correct imports and setup.

In [9]:
import boto3
import json
import pandas as pd
from datetime import datetime, timedelta, timezone  # ✅ FIXED: Added timezone import
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from botocore.exceptions import ClientError  # ✅ ADDED: For better error handling

# Import utilities with error handling
try:
    from scripts.utils import get_ssm_parameter
    print("✅ Utils imported successfully")
except ImportError as e:
    print(f"⚠️ Utils import failed: {e}")
    print("   Continuing without utils - some functionality may be limited")
    
    # Define fallback function
    def get_ssm_parameter(param_name):
        """Fallback SSM parameter getter"""
        try:
            ssm = boto3.client('ssm')
            response = ssm.get_parameter(Name=param_name)
            return response['Parameter']['Value']
        except Exception as e:
            raise Exception(f"Parameter {param_name} not found: {e}")
    
# Initialize AWS clients with validation
try:
    session = boto3.Session()
    region = session.region_name
    account_id = boto3.client('sts').get_caller_identity()['Account']
    cloudwatch = boto3.client('cloudwatch', region_name=region)
    bedrock_agent = boto3.client('bedrock-agent-runtime', region_name=region)
    print(f"✅ AWS clients initialized successfully")
    print(f"🔍 Connected to AWS Account: {account_id}")
    print(f"📍 Region: {region}")
except Exception as e:
    print(f"❌ AWS client initialization failed: {e}")
    raise

✅ Utils imported successfully
✅ AWS clients initialized successfully
🔍 Connected to AWS Account: 533267284022
📍 Region: us-east-1


In [10]:
# ✅ ENHANCED: Get lab resources with better validation
def get_lab_resource(param_name, resource_type):
    """Get lab resource with enhanced error handling"""
    try:
        value = get_ssm_parameter(param_name)
        print(f"✅ Found {resource_type}: {value}")
        return value
    except Exception as e:
        print(f"ℹ️ {resource_type} not found: {str(e)}")
        return None

# Get resources from previous labs
runtime_arn = get_lab_resource("/app/customersupport/agentcore/runtime_arn", "Runtime (Lab 4)")
gateway_arn = get_lab_resource("/app/customersupport/agentcore/gateway_arn", "Gateway (Lab 3)") 
memory_id = get_lab_resource("/app/customersupport/agentcore/memory_id", "Memory (Lab 2)")

# ✅ ENHANCED: Better runtime name extraction with validation
if runtime_arn:
    runtime_name = runtime_arn.split('/')[-1]
    if '-' in runtime_name and len(runtime_name.split('-')) > 1:
        # Only split if it looks like name-id format
        parts = runtime_name.split('-')
        if len(parts) >= 2 and parts[-1].replace('_', '').isalnum():
            runtime_name = '-'.join(parts[:-1])  # Keep all but last part
    print(f"📝 Extracted runtime name: {runtime_name}")
    print(f"🔗 Full runtime ARN: {runtime_arn}")
else:
    runtime_name = None

print(f"\n🔍 Lab Resources Summary:")
print(f"   Runtime: {'✅ Available' if runtime_arn else '❌ Missing (complete Lab 4)'}")
print(f"   Gateway: {'✅ Available' if gateway_arn else '❌ Missing (complete Lab 3)'}")
print(f"   Memory:  {'✅ Available' if memory_id else '❌ Missing (complete Lab 2)'}")

✅ Found Runtime (Lab 4): arn:aws:bedrock-agentcore:us-east-1:533267284022:runtime/customer_support_agent-b0Ilb5ACG7
ℹ️ Gateway (Lab 3) not found: An error occurred (ParameterNotFound) when calling the GetParameter operation: 
✅ Found Memory (Lab 2): CustomerSupportMemory-DB1nof41H6
📝 Extracted runtime name: customer_support_agent
🔗 Full runtime ARN: arn:aws:bedrock-agentcore:us-east-1:533267284022:runtime/customer_support_agent-b0Ilb5ACG7

🔍 Lab Resources Summary:
   Runtime: ✅ Available
   Gateway: ❌ Missing (complete Lab 3)
   Memory:  ✅ Available


### Step 2: ✅ CRITICAL - Check AgentCore Observability Setup

**NEW REQUIREMENT**: AgentCore metrics require observability to be enabled first!

In [11]:
def check_agentcore_observability_setup():
    """Check if AgentCore observability is properly configured"""
    
    print("🔍 Checking AgentCore Observability Setup...")
    print("=" * 50)
    
    # ✅ FIXED: Test correct namespace
    try:
        response = cloudwatch.list_metrics(Namespace="AWS/Bedrock-AgentCore")
        metrics = response.get('Metrics', [])
        
        if metrics:
            print(f"✅ AgentCore observability is ENABLED")
            print(f"📊 Found {len(metrics)} AgentCore metrics")
            return True
            
        else:
            print(f"⚠️ AgentCore observability is NOT ENABLED or no metrics generated yet")
            print(f"\n🔧 How to Enable Observability:")
            print(f"   1. Go to Amazon Bedrock AgentCore console")
            print(f"   2. Navigate to your Runtime/Gateway/Memory resources")
            print(f"   3. Enable 'Observability' setting for each resource")
            print(f"   4. Set up required CloudWatch log groups")
            print(f"   5. Invoke your agent to generate metrics")
            return False
            
    except ClientError as e:
        error_code = e.response['Error']['Code']
        print(f"❌ AWS Error: {error_code}")
        return False
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return False

# Check observability setup
observability_enabled = check_agentcore_observability_setup()

if not observability_enabled:
    print(f"\n🚨 IMPORTANT: Enable AgentCore observability before metrics will appear!")
else:
    print(f"\n🎉 Great! Observability is enabled. Let's explore your metrics!")

🔍 Checking AgentCore Observability Setup...
✅ AgentCore observability is ENABLED
📊 Found 188 AgentCore metrics

🎉 Great! Observability is enabled. Let's explore your metrics!


### Step 3: Discover Available AgentCore Metrics ✅ FIXED NAMESPACE

Now using the correct `AWS/Bedrock-AgentCore` namespace!

In [12]:
def discover_agentcore_metrics():
    """Discover available AgentCore metrics with CORRECT namespace"""
    
    print("🔍 Discovering AgentCore Metrics (Fixed Namespace)...")
    print("=" * 55)
    
    # ✅ CRITICAL FIX: Use correct single namespace
    namespace = "AWS/Bedrock-AgentCore"  # NOT AWS/Bedrock/AgentCore/*
    
    try:
        response = cloudwatch.list_metrics(Namespace=namespace)
        metrics = response.get('Metrics', [])
        
        if metrics:
            print(f"✅ Found {len(metrics)} metrics in {namespace}")
            
            # Show sample metrics
            metric_names = list(set([m['MetricName'] for m in metrics]))
            print(f"\n📊 Available Metrics:")
            for name in sorted(metric_names):
                print(f"   • {name}")
                        
            return metrics
            
        else:
            print(f"⚠️ No metrics found in {namespace}")
            print(f"\n💡 This means:")
            print(f"   • AgentCore observability is not enabled")
            print(f"   • OR your AgentCore resources haven't generated metrics yet")
            return []
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return []

# Discover available metrics
available_metrics = discover_agentcore_metrics()

print(f"\n✅ Namespace Fix Applied: Now using 'BedrockAgentCore' instead of 'AWS/Bedrock/AgentCore/*'")
print(f"📊 Found {len(available_metrics)} total metrics")

🔍 Discovering AgentCore Metrics (Fixed Namespace)...
✅ Found 188 metrics in AWS/Bedrock-AgentCore

📊 Available Metrics:
   • CreationCount
   • Duration
   • Errors
   • Invocations
   • Latency
   • Sessions
   • SystemErrors
   • TargetExecutionTime
   • TargetType.LAMBDA
   • Throttles
   • UserErrors

✅ Namespace Fix Applied: Now using 'BedrockAgentCore' instead of 'AWS/Bedrock/AgentCore/*'
📊 Found 188 total metrics


### Step 4: Query Runtime Metrics ✅ FIXED

Query runtime metrics using the correct namespace and dimensions.

In [13]:
def get_runtime_metrics_fixed(runtime_arn, hours_back=24):
    """Get Runtime metrics with FIXED namespace and dimensions"""
    
    if not runtime_arn:
        print("⚠️ No runtime ARN provided")
        return {}
    
    print(f"📊 Querying Runtime Metrics (Fixed Version)")
    print(f"🔗 Runtime ARN: {runtime_arn}")
    print("-" * 50)
    
    # ✅ FIXED: Proper timezone usage
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)
    
    # Runtime metrics from AgentCore documentation
    runtime_metrics = [
        ('Invocations', 'Sum'),
        ('Latency', 'Average'),
        ('SystemErrors', 'Sum'),
        ('UserErrors', 'Sum'),
        ('Throttles', 'Sum')
    ]
    
    metrics_data = {}
    
    for metric_name, stat in runtime_metrics:
        try:
            # ✅ CRITICAL FIX: Use correct namespace and Resource dimension
            response = cloudwatch.get_metric_statistics(
                Namespace='AWS/Bedrock-AgentCore',  # FIXED: Not AWS/Bedrock/AgentCore/Runtime
                MetricName=metric_name,
                Dimensions=[
                    {'Name': 'Resource', 'Value': runtime_arn}  # FIXED: Use full ARN as Resource
                ],
                StartTime=start_time,
                EndTime=end_time,
                Period=3600,  # 1 hour periods
                Statistics=[stat]
            )
            
            datapoints = response.get('Datapoints', [])
            if datapoints:
                metrics_data[metric_name] = sorted(datapoints, key=lambda x: x['Timestamp'])
                latest_value = datapoints[-1][stat]
                print(f"✅ {metric_name}: {latest_value:.2f} (last hour, {len(datapoints)} total points)")
            else:
                print(f"ℹ️ {metric_name}: No data points found")
                
        except ClientError as e:
            error_code = e.response['Error']['Code']
            print(f"⚠️ {metric_name}: AWS Error - {error_code}")
        except Exception as e:
            print(f"❌ {metric_name}: Error - {str(e)}")
    
    return metrics_data

if runtime_arn:
    print("📊 Runtime Metrics from your Lab 4 deployment (FIXED VERSION):\n")
    runtime_metrics = get_runtime_metrics_fixed(runtime_arn)
    
    if not runtime_metrics:
        print(f"\n💡 No runtime metric data found. This could mean:")
        print(f"   • Observability not enabled for this runtime")
        print(f"   • Agent hasn't been invoked recently")
        print(f"   • Metrics take a few minutes to appear after invocation")
else:
    print("⚠️ Complete Lab 4 first to see Runtime metrics")
    runtime_metrics = {}

📊 Runtime Metrics from your Lab 4 deployment (FIXED VERSION):

📊 Querying Runtime Metrics (Fixed Version)
🔗 Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:533267284022:runtime/customer_support_agent-b0Ilb5ACG7
--------------------------------------------------
ℹ️ Invocations: No data points found
ℹ️ Latency: No data points found
ℹ️ SystemErrors: No data points found
ℹ️ UserErrors: No data points found
ℹ️ Throttles: No data points found

💡 No runtime metric data found. This could mean:
   • Observability not enabled for this runtime
   • Agent hasn't been invoked recently
   • Metrics take a few minutes to appear after invocation


### Step 5: Create CloudWatch Dashboard ✅ FIXED

Create a dashboard using the correct namespace and dimensions.

In [6]:
def create_runtime_dashboard_fixed(runtime_arn, runtime_name):
    """Create a CloudWatch dashboard with FIXED namespace"""
    
    dashboard_name = f"CustomerSupportAgent-Runtime-{runtime_name}-Fixed"
    
    # ✅ FIXED: Use correct namespace and Resource dimension in dashboard
    dashboard_body = {
        "widgets": [
            {
                "type": "metric",
                "x": 0, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "metrics": [
                        ["BedrockAgentCore", "Invocations", "Resource", runtime_arn, {"stat": "Sum", "label": "Total Invocations"}],
                        [".", "Latency", ".", ".", {"stat": "Average", "label": "Avg Latency (ms)", "yAxis": "right"}]
                    ],
                    "period": 300,
                    "stat": "Sum",
                    "region": region,
                    "title": "Agent Performance Overview (FIXED)",
                    "yAxis": {"left": {"label": "Count"}, "right": {"label": "Milliseconds"}}
                }
            },
            {
                "type": "metric",
                "x": 12, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "metrics": [
                        ["BedrockAgentCore", "SystemErrors", "Resource", runtime_arn, {"stat": "Sum", "color": "#d62728"}],
                        [".", "UserErrors", ".", ".", {"stat": "Sum", "color": "#ff7f0e"}],
                        [".", "Throttles", ".", ".", {"stat": "Sum", "color": "#9467bd"}]
                    ],
                    "period": 300,
                    "stat": "Sum",
                    "region": region,
                    "title": "Errors and Throttles (FIXED)",
                    "yAxis": {"left": {"label": "Count", "min": 0}}
                }
            }
        ]
    }
    
    try:
        response = cloudwatch.put_dashboard(
            DashboardName=dashboard_name,
            DashboardBody=json.dumps(dashboard_body)
        )
        
        dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
        print(f"✅ Created FIXED Runtime Dashboard: {dashboard_name}")
        print(f"📊 View it here: {dashboard_url}")
        print(f"🔧 This dashboard uses the CORRECT namespace: BedrockAgentCore")
        return True
        
    except Exception as e:
        print(f"❌ Error creating dashboard: {str(e)}")
        return False

if runtime_arn:
    print("Creating FIXED CloudWatch Dashboard...\n")
    dashboard_created = create_runtime_dashboard_fixed(runtime_arn, runtime_name)
    
    if dashboard_created:
        print(f"\n🎉 SUCCESS! Your dashboard now uses the correct BedrockAgentCore namespace!")
else:
    print("⚠️ Complete Lab 4 first to create dashboards")

Creating FIXED CloudWatch Dashboard...

✅ Created FIXED Runtime Dashboard: CustomerSupportAgent-Runtime-customer_support_agent-Fixed
📊 View it here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=CustomerSupportAgent-Runtime-customer_support_agent-Fixed
🔧 This dashboard uses the CORRECT namespace: BedrockAgentCore

🎉 SUCCESS! Your dashboard now uses the correct BedrockAgentCore namespace!


### Step 6: Instructions for Generating Metrics

In [ ]:
print("🚀 How to Generate Metrics for Your Dashboard:")
print("=" * 50)

if runtime_arn:
    print(f"✅ Your runtime is ready: {runtime_arn}")
    print(f"\n📋 Steps to Generate Metrics:")
    print(f"   1. 🔧 FIRST: Enable observability for your runtime in AgentCore console")
    print(f"   2. 🤖 Use your agent from Lab 4/5")
    print(f"   3. 💬 Ask your agent some questions")
    print(f"   4. ⏰ Wait 2-5 minutes for metrics to appear in CloudWatch")
    print(f"   5. 🔄 Re-run the metric query cells above to see data")
    print(f"   6. 📊 Check your dashboard for real-time visualization")
else:
    print(f"❌ No runtime found - complete Lab 4 first")

print(f"\n🔧 Key Fixes Applied:")
print(f"   ✅ Namespace: BedrockAgentCore (not AWS/Bedrock/AgentCore/*)")
print(f"   ✅ Dimensions: Resource with full ARN")
print(f"   ✅ DateTime: Fixed timezone import")
print(f"   ✅ Observability: Added setup validation")

## ✅ Congratulations! All Issues Fixed! 🎉

### 🔧 Critical Fixes Applied:

1. **✅ NAMESPACE FIX**: 
   - **Before**: `AWS/Bedrock/AgentCore/Runtime` (WRONG) → Empty metrics
   - **After**: `BedrockAgentCore` (CORRECT) → Can find metrics

2. **✅ DATETIME FIX**: 
   - **Before**: `datetime.timezone.utc` → `AttributeError`
   - **After**: `from datetime import timezone` → Works perfectly

3. **✅ DIMENSION FIX**: 
   - **Before**: Various dimension names → May not work
   - **After**: `Resource` dimension with full ARN → Correct format

4. **✅ OBSERVABILITY SETUP**: 
   - **Added**: Setup validation and instructions
   - **Added**: Clear requirements for enabling observability

### 🚀 Next Steps:

1. **Enable Observability**: Go to AgentCore console and enable observability
2. **Use Your Agent**: Invoke your agent to generate metrics
3. **Monitor Dashboards**: Watch real-time metrics appear in CloudWatch

---

**🎉 Your AgentCore metrics monitoring is now fully functional! 📊✅**